# Notebook 02 — Landing Zone CSV → Delta Lake (Bronze)

Lê os **11 arquivos CSV** do bucket `landing-zone` e converte cada um para o formato **Delta Lake** no bucket `bronze`.

**Fluxo:**
```
MinIO s3a://landing-zone/{tabela}.csv
    ──Spark Read──► DataFrame
    ──Delta Write──► MinIO s3a://bronze/{tabela}/
```

## 1. Imports e configuração

In [ ]:
import os
import boto3
from botocore.client import Config
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip, DeltaTable

load_dotenv()

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT', 'http://localhost:9020')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET', 'landing-zone')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET', 'bronze')

# Remove o prefixo http:// para o endpoint S3A
S3A_ENDPOINT = MINIO_ENDPOINT.replace('http://', '').replace('https://', '')

print(f'Landing Zone: s3a://{LANDING_BUCKET}')
print(f'Bronze: s3a://{BRONZE_BUCKET}')

## 2. Criar SparkSession com Delta Lake + MinIO (S3A)

In [ ]:
builder = (
    SparkSession.builder
    .appName('landing-to-bronze-delta')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    # Configurações S3A para MinIO
    .config('spark.hadoop.fs.s3a.endpoint', f'http://{S3A_ENDPOINT}')
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.jars.packages',
            'io.delta:delta-spark_2.12:3.2.0,'
            'org.apache.hadoop:hadoop-aws:3.3.4,'
            'com.amazonaws:aws-java-sdk-bundle:1.12.262')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')

print(f'Spark {spark.version} iniciado!')

## 3. Criar bucket bronze no MinIO

In [ ]:
s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

existing = [b['Name'] for b in s3.list_buckets()['Buckets']]
if BRONZE_BUCKET not in existing:
    s3.create_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket criado: {BRONZE_BUCKET}')
else:
    print(f'Bucket já existe: {BRONZE_BUCKET}')

## 4. Converter cada CSV para Delta Lake

In [ ]:
# Lista os CSVs disponíveis no landing-zone
response = s3.list_objects_v2(Bucket=LANDING_BUCKET)
csvs = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.csv')]

print(f'{len(csvs)} arquivos CSV encontrados no landing-zone:\n')

resultados = []

for csv_key in csvs:
    tabela = csv_key.replace('.csv', '')
    landing_path = f's3a://{LANDING_BUCKET}/{csv_key}'
    bronze_path  = f's3a://{BRONZE_BUCKET}/{tabela}'

    # Lê o CSV
    df = spark.read.csv(landing_path, header=True, inferSchema=True)

    # Escreve como Delta Lake (overwrite para re-execuções)
    df.write.format('delta').mode('overwrite').save(bronze_path)

    count = df.count()
    resultados.append({'tabela': tabela, 'registros': count, 'path': bronze_path})
    print(f'  {tabela:15s} → {count:5d} registros → {bronze_path}')

print(f'\nTotal: {sum(r["registros"] for r in resultados)} registros convertidos para Delta Lake')

## 5. Validação — verificar tabelas Delta

In [ ]:
print('=== Validação das tabelas Delta no bucket bronze ===')
for r in resultados:
    tabela = r['tabela']
    path   = r['path']

    is_delta = DeltaTable.isDeltaTable(spark, path)
    count    = spark.read.format('delta').load(path).count()

    status = 'OK' if is_delta and count == r['registros'] else 'ERRO'
    print(f'  [{status}] {tabela:15s} | isDelta={is_delta} | {count} registros')

## 6. Inspecionar schema e dados de uma tabela

In [ ]:
# Exemplo com a tabela apolice
df_apolice = spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/apolice')

print('Schema da tabela apolice:')
df_apolice.printSchema()

print('\nPrimeiros registros:')
df_apolice.show(5, truncate=False)